# Root2Raj Credit Linkage Under Uncertainty
### How much does the matching rule change the story?

**Root2Raj · Ruturajsinh Zala · Reproducible public-data contribution**

AI-assisted portfolio work. This is a reproducible experiment, not employment, client work, or a claim of independent coding proficiency.

[Original revenue audit](https://www.kaggle.com/code/ruturajsinhzala/root2raj-retail-revenue-and-credit-audit) · [Reusable source and tests](https://github.com/Root2Raj-Zala/root2raj-retail-operations/tree/main/contributions) · [Public aggregate dataset](https://www.kaggle.com/datasets/ruturajsinhzala/root2raj-retail-audit-evidence)

**Source:** Chen, D. (2015), *Online Retail*, UCI. [Dataset](https://archive.ics.uci.edu/dataset/352/online%2Bretail) · [DOI](https://doi.org/10.24432/C5BW33) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Historical UK retailer records, December 2010–December 2011.

## Reproducible input and explicit cleaning
CPU is sufficient. Enable Internet for the original UCI download, or attach the unchanged Online Retail.xlsx. We verify the workbook SHA-256 and 541,909-row schema before analysis. Exact duplicates are removed as an explicit assumption, consistent with the original audit. Identical lines could represent genuine repeats.

A sale has positive quantity, positive price, a valid invoice/date, and a non-C invoice. A credit has negative quantity under a C invoice. Other rows are excluded from these experiments. No customer identifiers appear in outputs.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, urllib.request, zipfile, platform, unittest
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
ROOT=Path("/kaggle/working/root2raj-credit-linkage-uncertainty") if Path("/kaggle/working").is_dir() else Path(os.environ.get("ROOT2RAJ_HOME",str(Path.cwd()/"root2raj-credit-linkage-uncertainty-run")))
DATA=Path("/kaggle/temp/root2raj-input") if Path("/kaggle/working").is_dir() else ROOT/"data"
OUT=ROOT/"outputs"
sys.dont_write_bytecode=True
DATA.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
EXPECTED_HASH="43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d"
attached=sorted(Path("/kaggle/input").rglob("Online Retail.xlsx")) if Path("/kaggle/input").is_dir() else []
source=attached[0] if attached else DATA/"Online Retail.xlsx"
if not source.exists():
    archive=DATA/"uci-source.zip"
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/352/online%2Bretail.zip",archive)
    with zipfile.ZipFile(archive) as z:
        names=[n for n in z.namelist() if Path(n).name=="Online Retail.xlsx"]
        if len(names)!=1: raise ValueError("Expected one UCI workbook")
        source.write_bytes(z.read(names[0]))
source_hash=hashlib.sha256(source.read_bytes()).hexdigest()
assert source_hash==EXPECTED_HASH,"Source workbook changed"
raw=pd.read_excel(source,engine="openpyxl")
assert set(raw.columns)=={"InvoiceNo","StockCode","Description","Quantity","InvoiceDate","UnitPrice","CustomerID","Country"}
assert len(raw)==541909
duplicate_count=int(raw.duplicated().sum())
df=raw.drop_duplicates().copy()
df.InvoiceNo=df.InvoiceNo.astype("string").str.strip()
df.StockCode=df.StockCode.astype("string").str.strip()
df.InvoiceDate=pd.to_datetime(df.InvoiceDate,errors="coerce")
valid=(df.InvoiceDate.notna() & df.InvoiceNo.notna() & np.isfinite(df.Quantity) & np.isfinite(df.UnitPrice) & (df.UnitPrice>0) & (df.Quantity!=0))
cancel=df.InvoiceNo.str.upper().str.startswith("C").fillna(False)
df["kind"]="excluded"
df.loc[valid & ~cancel & (df.Quantity>0),"kind"]="sale"
df.loc[valid & cancel & (df.Quantity<0),"kind"]="credit"
assert (df.loc[df.kind.isin(["sale","credit"]),"Quantity"]%1==0).all()
print("Verified source:",source_hash)
display(pd.DataFrame([{"raw_rows":len(raw),"duplicate_rows_removed":duplicate_count,"sale_rows":int((df.kind=="sale").sum()),"credit_rows":int((df.kind=="credit").sum())}]))

## Reusable implementation
The module below is written into the run directory and imported. It is also included in the evidence ZIP, together with meaningful synthetic tests. The complete implementation is visible so readers can inspect or adapt every assumption.

In [ ]:
module_source = r'''"""Capacity-constrained candidate credit linkage; no claim of ground truth."""
from collections import deque, defaultdict
from datetime import timedelta
import pandas as pd

def match_events(events, window_days=90, policy="FIFO"):
    if policy not in {"FIFO","LIFO"} or window_days <= 0:
        raise ValueError("Expected FIFO/LIFO and a positive lookback")
    # kind 0 = credit, kind 1 = sale: same-time sales cannot fund credits.
    events=sorted(events,key=lambda e:(e[0],e[1],e[4]))
    lots=deque()
    matched=unmatched=lag_value=matched_value=0.0
    buckets=defaultdict(float)
    monthly=defaultdict(lambda:[0.0,0.0])
    credit_units=credit_value=0.0
    for time,kind,quantity,price,position in events:
        if quantity<=0 or price<=0 or kind not in (0,1):
            raise ValueError("Events require positive magnitudes and prices")
        while lots and lots[0][0] < time-timedelta(days=window_days):
            lots.popleft()
        if kind==1:
            lots.append([time,float(quantity)])
            continue
        remaining=float(quantity)
        credit_units+=remaining
        credit_value+=remaining*price
        month=time.strftime("%Y-%m")
        monthly[month][1]+=remaining*price
        while lots and remaining>0:
            lot=lots[0] if policy=="FIFO" else lots[-1]
            lag=(time-lot[0]).total_seconds()/86400
            assert 0 < lag <= window_days
            taken=min(remaining,lot[1])
            value=taken*price
            matched+=taken
            matched_value+=value
            lag_value+=lag*value
            monthly[month][0]+=value
            bucket="<1 day" if lag<1 else "1-7 days" if lag<=7 else "7-30 days" if lag<=30 else "30-90 days" if lag<=90 else "90-365 days"
            buckets[bucket]+=value
            remaining-=taken
            lot[1]-=taken
            if lot[1]==0:
                lots.popleft() if policy=="FIFO" else lots.pop()
        unmatched+=remaining
    assert abs(matched+unmatched-credit_units)<1e-6
    assert matched_value<=credit_value+1e-6
    return dict(matched_units=matched,unmatched_units=unmatched,matched_value=matched_value,
                credit_value=credit_value,lag_value=lag_value,buckets=dict(buckets),monthly=dict(monthly))

def prepare_groups(frame):
    required={"CustomerID","StockCode","UnitPrice","InvoiceDate","Quantity","kind"}
    if not required.issubset(frame.columns):
        raise ValueError("Missing required columns")
    eligible=frame[frame.CustomerID.notna() & frame.kind.isin(["sale","credit"])].copy()
    eligible["position"]=range(len(eligible))
    eligible["event_kind"]=(eligible.kind=="sale").astype(int)
    eligible=eligible.sort_values(["InvoiceDate","event_kind","position"],kind="stable")
    groups=[]
    for _,g in eligible.groupby(["CustomerID","StockCode","UnitPrice"],sort=False):
        if not (g.event_kind==0).any():
            continue
        groups.append([(t.to_pydatetime(),int(k),abs(float(q)),float(p),int(i))
                       for t,k,q,p,i in g[["InvoiceDate","event_kind","Quantity","UnitPrice","position"]].itertuples(index=False,name=None)])
    return groups

def run_linkage(frame,windows=(7,30,90,365)):
    groups=prepare_groups(frame)
    credits=frame[frame.kind=="credit"]
    total=float((-credits.Quantity*credits.UnitPrice).sum())
    identified=float((-credits.loc[credits.CustomerID.notna(),"Quantity"]*credits.loc[credits.CustomerID.notna(),"UnitPrice"]).sum())
    summary=[]; lag_rows=[]; monthly_rows=[]
    for window in windows:
        for policy in ("FIFO","LIFO"):
            parts=[match_events(g,window,policy) for g in groups]
            value=sum(p["matched_value"] for p in parts)
            eligible_value=sum(p["credit_value"] for p in parts)
            assert abs(eligible_value-identified)<1e-5
            buckets=defaultdict(float); months=defaultdict(lambda:[0.,0.])
            for part in parts:
                for b,v in part["buckets"].items(): buckets[b]+=v
                for m,vals in part["monthly"].items():
                    for i in (0,1): months[m][i]+=vals[i]
            summary.append(dict(window_days=window,policy=policy,matched_value_gbp=value,
                unmatched_identified_value_gbp=identified-value,missing_id_credit_value_gbp=total-identified,
                all_credit_value_gbp=total,identified_credit_value_gbp=identified,
                coverage_all_pct=100*value/total,coverage_identified_pct=100*value/identified,
                value_weighted_lag_days=sum(p["lag_value"] for p in parts)/value if value else 0))
            for b in ["<1 day","1-7 days","7-30 days","30-90 days","90-365 days"]:
                lag_rows.append(dict(window_days=window,policy=policy,lag_bucket=b,matched_value_gbp=buckets[b]))
            all_months=(-credits.Quantity*credits.UnitPrice).groupby(credits.InvoiceDate.dt.strftime("%Y-%m")).sum()
            for month,all_value in all_months.items():
                vals=months[month]
                monthly_rows.append(dict(window_days=window,policy=policy,month=month,
                    matched_value_gbp=vals[0],identified_credit_value_gbp=vals[1],all_credit_value_gbp=float(all_value),
                    coverage_all_pct=100*vals[0]/all_value if all_value else 0))
    return pd.DataFrame(summary),pd.DataFrame(lag_rows),pd.DataFrame(monthly_rows)
'''
(ROOT/'credit_linkage.py').write_text(module_source,encoding="utf-8")
sys.path.insert(0,str(ROOT))
from credit_linkage import *
NAVY,TEAL,CORAL,GREY="#162A43","#087F8C","#D56343","#66758A"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.spines.top":False,"axes.spines.right":False,"text.color":NAVY,"axes.labelcolor":NAVY,"font.size":11,"axes.titlesize":15,"axes.titleweight":"bold","figure.dpi":130})
pd.set_option("display.float_format",lambda v:f"{v:,.2f}")
def chart(fig,name):
    fig.savefig(OUT/name,dpi=170,bbox_inches="tight")
    plt.show(); plt.close(fig)

## Experiment: can credits be linked to earlier sale capacity?
For each identified customer × stock code × exact unit-price key, earlier positive units form available sale capacity. Each credit consumes that capacity at most once. Matching requires a strictly earlier sale within an inclusive lookback window. Same-timestamp sales cannot fund a credit.

We compare oldest-sale-first (FIFO) and newest-sale-first (LIFO), with 7, 30, 90 and 365-day windows. Stable source position breaks ties. Missing customer IDs are never guessed. Partial matching is allowed.

**This is candidate linkage, not a verified return rate.** There is no original-invoice reference or return-reason field. Exact prices can miss discounts or corrections; matching keys can also link an unrelated earlier purchase. The start of the extract hides prior sales. Special stock codes and non-merchandise credits remain in scope, preserving the original value definition.

In [ ]:
summary,lag_distribution,monthly_linkage=run_linkage(df)
summary.to_csv(OUT/"scenario_summary.csv",index=False)
lag_distribution.to_csv(OUT/"lag_distribution.csv",index=False)
monthly_linkage.to_csv(OUT/"monthly_linkage.csv",index=False)
display(summary)
checks={
 "source_rows_verified":len(raw)==541909,
 "source_hash_verified":source_hash==EXPECTED_HASH,
 "coverage_bounded":bool(summary.coverage_all_pct.between(0,100).all()),
 "value_conservation":bool(np.allclose(summary.matched_value_gbp+summary.unmatched_identified_value_gbp+summary.missing_id_credit_value_gbp,summary.all_credit_value_gbp)),
 "lag_bucket_reconciliation":bool(np.allclose(lag_distribution.groupby(["window_days","policy"]).matched_value_gbp.sum().sort_index(),summary.set_index(["window_days","policy"]).matched_value_gbp.sort_index())),
 "monthly_reconciliation":bool(np.allclose(monthly_linkage.groupby(["window_days","policy"]).matched_value_gbp.sum().sort_index(),summary.set_index(["window_days","policy"]).matched_value_gbp.sort_index()))}
assert all(checks.values()),checks

## Sensitivity: coverage depends on the matching rule
The all-credit denominator includes credits with missing customer IDs. The identified-credit denominator excludes them. Reporting both prevents an apparently strong matching percentage from hiding the unidentifiable part of the ledger.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for policy,color in [("FIFO",TEAL),("LIFO",CORAL)]:
    g=summary[summary.policy==policy]
    for ax,col,label in [(axes[0],"coverage_all_pct","All credit value"),(axes[1],"coverage_identified_pct","Identified credit value")]:
        ax.plot(g.window_days,g[col],marker="o",label=policy,color=color)
        ax.set(xlabel="Lookback window (days)",ylabel="Candidate coverage (%)",title=label)
        ax.set_xticks([7,30,90,365]); ax.set_ylim(0,100); ax.legend()
fig.suptitle("Candidate matching is a sensitivity analysis",y=1.04,weight="bold")
chart(fig,"01-coverage-sensitivity.png")

## Timing uncertainty remains after matching
At a 90-day lookback, the two policies can assign the same credit to different earlier sale lots. The chart shows matched GBP by elapsed-time bucket, not elapsed time to a confirmed return.

In [ ]:
g=lag_distribution[lag_distribution.window_days==90].pivot(index="lag_bucket",columns="policy",values="matched_value_gbp").reindex(["<1 day","1-7 days","7-30 days","30-90 days","90-365 days"])
fig,ax=plt.subplots(figsize=(11,4.5))
(g/1000).plot.bar(ax=ax,color=[TEAL,CORAL],rot=0)
ax.set(title="90-day candidate linkage: timing changes by policy",ylabel="Matched credit value (£000)",xlabel="Elapsed time since candidate sale")
chart(fig,"02-lag-uncertainty.png")
fig,ax=plt.subplots(figsize=(11,4.5))
for policy,color in [("FIFO",TEAL),("LIFO",CORAL)]:
    g=monthly_linkage[(monthly_linkage.window_days==90)&(monthly_linkage.policy==policy)]
    ax.plot(g.month,g.coverage_all_pct,marker="o",color=color,label=policy)
ax.set(title="Monthly candidate coverage at 90 days",ylabel="Share of all monthly credit value (%)",xlabel="")
ax.tick_params(axis="x",rotation=45); ax.legend()
chart(fig,"03-monthly-coverage.png")

## Decision note
Use this experiment to identify how much credit value can be associated with available historical sale capacity, and how strongly a timing report depends on a chosen rule. Investigate unmatched value with better ledger references; do not label it fraud, defects, or customer behavior.

December 2011 is partial. Early months are especially affected by missing pre-extract history. A larger window increases opportunity for plausible but unrelated links. Policy agreement is not proof of correctness.

In [ ]:
r90=summary[summary.window_days==90]
spread=float(r90.value_weighted_lag_days.max()-r90.value_weighted_lag_days.min())
missing=float(summary.iloc[0].missing_id_credit_value_gbp)
display(Markdown(f"**Measured result:** At 90 days, candidate coverage spans **{r90.coverage_all_pct.min():.2f}%–{r90.coverage_all_pct.max():.2f}% of all credit value**. FIFO/LIFO value-weighted lag differs by **{spread:.2f} days**. **£{missing:,.2f}** of credits lack customer IDs and remain unlinked."))
results={"source_sha256":source_hash,"scenario_count":len(summary),"coverage_90_days_all_pct":[float(r90.coverage_all_pct.min()),float(r90.coverage_all_pct.max())],"lag_policy_spread_days_90":spread,"missing_id_credit_value_gbp":missing,"interpretation":"Capacity-constrained candidates, not verified returns"}
(OUT/"results.json").write_text(json.dumps(results,indent=2),encoding="utf-8")
(OUT/"DATA_DICTIONARY.md").write_text("""# Evidence dictionary
scenario_summary.csv: one row per lookback window and policy. matched_value_gbp is allocated candidate credit value; unmatched_identified_value_gbp is known-ID credit value without allocated capacity; missing_id_credit_value_gbp is unidentifiable credit value. coverage_all_pct divides matched value by all credits; coverage_identified_pct divides by known-ID credits. value_weighted_lag_days averages candidate lags weighted by allocated GBP.
lag_distribution.csv: matched credit GBP by scenario and inclusive upper-bound lag bucket.
monthly_linkage.csv: scenario × credit month; denominators refer to credits recorded in that month. December 2011 is partial.
No invoice or customer identifiers are exported. Amounts are historical GBP, with floating-point calculations checked to tolerances.
""",encoding="utf-8")

## Data architecture: contracts, lineage and a queryable gold layer
The unchanged workbook is the bronze snapshot; explicit deduplication and classification form the silver layer. Experiment-specific aggregate tables form gold data products. This is a local reference architecture, not a deployed warehouse or an orchestration service.

Version 1.0.0 contracts declare table grain, types, nullability and nonnegative measures. SQLite STRICT tables enforce those types and primary keys. The notebook checks database integrity, row counts, and duplicate-key rejection. A lineage manifest records the workbook hash, implementation hash, CSV hashes and transformation edges. Customer and invoice IDs are excluded from the gold schema. Raw and silver layers stay in memory or source storage; only gold is exported.

In [ ]:
architecture_source = r'''"""Export versioned gold data products with lineage and executable SQL contracts."""
from pathlib import Path
from contextlib import closing
import hashlib,json,sqlite3
import pandas as pd
import numpy as np

CONTRACT_VERSION="1.0.0"

def export_data_product(out,tables,primary_keys,source_hash,implementation,experiment):
    out=Path(out)
    schema={}; ddl=[]; checks={}
    assert all("CustomerID" not in t.columns and "InvoiceNo" not in t.columns for t in tables.values())
    for name,table in tables.items():
        if not name.replace("_","").isalnum(): raise ValueError("Invalid table identifier")
        keys=primary_keys[name]
        if not set(keys).issubset(table.columns): raise ValueError("Missing primary key")
        if table.empty or table.isna().any().any() or table.duplicated(keys).any():
            raise ValueError(f"{name}: empty table, null values or duplicate primary key")
        fields=[]; clauses=[]
        for col in table.columns:
            quoted='"'+str(col).replace('"','""')+'"'
            dtype=table[col].dtype
            sql_type="INTEGER" if pd.api.types.is_integer_dtype(dtype) or pd.api.types.is_bool_dtype(dtype) else "REAL" if pd.api.types.is_numeric_dtype(dtype) else "TEXT"
            if sql_type in ("INTEGER","REAL") and not np.isfinite(table[col].astype(float)).all():
                raise ValueError(f"{name}.{col}: non-finite numeric value")
            nonnegative=sql_type!="TEXT" and (col.endswith("_gbp") or "units" in col or "wape" in col or "coverage" in col or col=="window_days")
            if nonnegative and (table[col]<0).any(): raise ValueError(f"{name}.{col}: negative value")
            clause=f"{quoted} {sql_type} NOT NULL"
            if nonnegative: clause+=f" CHECK ({quoted} >= 0)"
            clauses.append(clause)
            fields.append(dict(name=str(col),sql_type=sql_type,nullable=False,nonnegative=nonnegative))
        key_sql=", ".join('"'+str(k).replace('"','""')+'"' for k in keys)
        statement=f'CREATE TABLE "{name}" (\n  '+",\n  ".join(clauses)+f",\n  PRIMARY KEY ({key_sql})\n) STRICT;"
        ddl.append(statement)
        schema[name]={"grain":keys,"row_count":len(table),"fields":fields}
    with closing(sqlite3.connect(out/"gold.sqlite")) as con:
        for name,table in tables.items():
            con.execute(f'DROP TABLE IF EXISTS "{name}"')
            con.execute(ddl[list(tables).index(name)])
            normalized=table.copy()
            for col in normalized.columns:
                if pd.api.types.is_datetime64_any_dtype(normalized[col].dtype):
                    normalized[col]=normalized[col].dt.strftime("%Y-%m-%d")
            normalized.to_sql(name,con,index=False,if_exists="append")
            count=con.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]
            assert count==len(table)
            checks[name+"_sql_row_count"]=True
            # The primary-key rejection is tested against a real exported row.
            quoted_columns=", ".join('"'+str(c).replace('"','""')+'"' for c in table.columns)
            con.execute("SAVEPOINT contract_probe")
            try:
                con.execute(f'INSERT INTO "{name}" ({quoted_columns}) SELECT {quoted_columns} FROM "{name}" LIMIT 1')
            except sqlite3.IntegrityError:
                checks[name+"_duplicate_key_rejected"]=True
            else:
                raise AssertionError("SQL primary-key contract failed")
            finally:
                con.execute("ROLLBACK TO contract_probe")
                con.execute("RELEASE contract_probe")
        checks["sql_integrity_check"]=con.execute("PRAGMA integrity_check").fetchone()[0]=="ok"
        con.commit()
    (out/"gold_schema.sql").write_text("\n\n".join(ddl),encoding="utf-8")
    contracts={"schema_version":CONTRACT_VERSION,"layer":"gold","tables":schema,
        "freshness":"Static historical snapshot; no scheduled ingestion or live freshness claim",
        "breaking_change_rule":"Increase major version if a field type, primary key, target, or denominator changes"}
    (out/"data_contracts.json").write_text(json.dumps(contracts,indent=2),encoding="utf-8")
    lineage={"manifest_version":CONTRACT_VERSION,"experiment":experiment,
        "source":{"name":"UCI Online Retail","doi":"10.24432/C5BW33","license":"CC BY 4.0","workbook_sha256":source_hash},
        "implementation_sha256":hashlib.sha256(implementation.encode()).hexdigest(),
        "layers":[
            {"id":"bronze","grain":"Original invoice line","operation":"Unchanged workbook; source hash and schema verification"},
            {"id":"silver","grain":"Deduplicated invoice line","operation":"Exact duplicate removal and explicit sale/credit classification"},
            {"id":"experiment","grain":"Defined in notebook","operation":experiment},
            {"id":"gold","grain":"Declared primary keys","operation":"Aggregate evidence exported under executable SQLite STRICT contracts"}],
        "edges":[["bronze","silver"],["silver","experiment"],["experiment","gold"]],
        "privacy":"No invoice or customer identifiers in gold outputs; source workbook is not redistributed",
        "tables":[{"name":name,"csv_sha256":hashlib.sha256((out/(name+".csv")).read_bytes()).hexdigest(),"rows":len(t)} for name,t in tables.items()]}
    (out/"lineage.json").write_text(json.dumps(lineage,indent=2),encoding="utf-8")
    return checks
'''
(ROOT/"evidence_contracts.py").write_text(architecture_source,encoding="utf-8")
from evidence_contracts import export_data_product
architecture_checks=export_data_product(OUT,{"scenario_summary":summary,"lag_distribution":lag_distribution,"monthly_linkage":monthly_linkage},{"scenario_summary":["window_days","policy"],"lag_distribution":["window_days","policy","lag_bucket"],"monthly_linkage":["window_days","policy","month"]},source_hash,module_source+architecture_source,'Capacity-constrained credit candidates under eight predefined matching scenarios')
checks.update(architecture_checks)
display(pd.DataFrame([architecture_checks]))
print("Exported gold.sqlite, gold_schema.sql, data_contracts.json and lineage.json")

## Synthetic tests and artifact export
These tests exercise behavior that can otherwise produce plausible but wrong results. Passing them supports implementation correctness under the stated definitions; it cannot establish that those definitions match the retailer's real processes.

In [ ]:
test_source = r'''import unittest
from datetime import datetime,timedelta
from credit_linkage import match_events,prepare_groups
import pandas as pd

class CreditTests(unittest.TestCase):
    def event(self,day,kind,q,price=2,pos=0):
        return (datetime(2020,1,1)+timedelta(days=day),kind,q,price,pos)
    def test_partial_and_capacity(self):
        result=match_events([self.event(0,1,5),self.event(1,0,8),self.event(2,0,3)])
        self.assertEqual(result["matched_units"],5)
        self.assertEqual(result["unmatched_units"],6)
    def test_future_sale(self):
        self.assertEqual(match_events([self.event(0,0,3),self.event(1,1,3)])["matched_units"],0)
    def test_same_timestamp(self):
        self.assertEqual(match_events([self.event(0,1,3),self.event(0,0,3)])["matched_units"],0)
    def test_window_boundary(self):
        self.assertEqual(match_events([self.event(0,1,3),self.event(7,0,3)],7)["matched_units"],3)
        self.assertEqual(match_events([self.event(0,1,3),self.event(7.01,0,3)],7)["matched_units"],0)
    def test_fifo_lifo_lag(self):
        events=[self.event(0,1,2),self.event(5,1,2),self.event(6,0,2)]
        self.assertEqual(match_events(events,90,"FIFO")["lag_value"],24)
        self.assertEqual(match_events(events,90,"LIFO")["lag_value"],4)
    def test_key_isolation_and_missing_id(self):
        f=pd.DataFrame([
            [1,"A",2,"2020-01-01",5,"sale"],
            [1,"A",3,"2020-01-02",-1,"credit"],
            [2,"A",2,"2020-01-02",-1,"credit"],
            [1,"B",2,"2020-01-02",-1,"credit"],
            [None,"A",2,"2020-01-02",-1,"credit"]],
            columns=["CustomerID","StockCode","UnitPrice","InvoiceDate","Quantity","kind"])
        f.InvoiceDate=pd.to_datetime(f.InvoiceDate)
        self.assertEqual(sum(match_events(g)["matched_units"] for g in prepare_groups(f)),0)
        self.assertEqual(len(prepare_groups(f)),3)
    def test_invalid_inputs(self):
        with self.assertRaises(ValueError): match_events([],0)
        with self.assertRaises(ValueError): match_events([] ,90,"random")
        with self.assertRaises(ValueError): match_events([self.event(0,1,-3)])
'''
(ROOT/'test_credit_linkage.py').write_text(test_source,encoding="utf-8")
exec(compile(test_source,'test_credit_linkage.py',"exec"))
architecture_tests = r'''import unittest,tempfile,os,json,sqlite3
from pathlib import Path
from contextlib import closing
import pandas as pd
from evidence_contracts import export_data_product

class ContractTests(unittest.TestCase):
    def setUp(self):
        self.temp=tempfile.TemporaryDirectory(dir=os.environ.get("ROOT2RAJ_TEST_TEMP",os.environ.get("TEMP",".")))
        self.out=Path(self.temp.name)
    def tearDown(self):
        self.temp.cleanup()
    def export(self,table):
        table.to_csv(self.out/"facts.csv",index=False)
        return export_data_product(self.out,{"facts":table},{"facts":["key"]},"a"*64,"verified implementation","Synthetic contract test")
    def test_valid_export_and_database_key_enforcement(self):
        checks=self.export(pd.DataFrame({"key":["A","B"],"units":[2,3]}))
        self.assertTrue(all(checks.values()))
        with closing(sqlite3.connect(self.out/"gold.sqlite")) as con:
            self.assertEqual(con.execute("SELECT SUM(units) FROM facts").fetchone()[0],5)
            with self.assertRaises(sqlite3.IntegrityError):
                con.execute("INSERT INTO facts VALUES ('C',-1)")
            with self.assertRaises(sqlite3.IntegrityError):
                con.execute("INSERT INTO facts VALUES ('C',NULL)")
            with self.assertRaises(sqlite3.IntegrityError):
                con.execute("INSERT INTO facts VALUES ('C','wrong type')")
        contracts=json.loads((self.out/"data_contracts.json").read_text())
        self.assertEqual(contracts["schema_version"],"1.0.0")
        lineage=json.loads((self.out/"lineage.json").read_text())
        self.assertEqual(len(lineage["tables"][0]["csv_sha256"]),64)
    def test_duplicate_primary_key_rejected(self):
        with self.assertRaises(ValueError): self.export(pd.DataFrame({"key":["A","A"],"units":[1,2]}))
    def test_null_measure_rejected(self):
        with self.assertRaises(ValueError): self.export(pd.DataFrame({"key":["A"],"units":[None]}))
    def test_negative_measure_rejected(self):
        with self.assertRaises(ValueError): self.export(pd.DataFrame({"key":["A"],"units":[-1]}))
    def test_nonfinite_measure_rejected(self):
        with self.assertRaises(ValueError): self.export(pd.DataFrame({"key":["A"],"units":[float("inf")]}))
'''
(ROOT/"test_evidence_contracts.py").write_text(architecture_tests,encoding="utf-8")
os.environ["ROOT2RAJ_TEST_TEMP"]=str(ROOT)
exec(compile(architecture_tests,"test_evidence_contracts.py","exec"))
suite=unittest.TestSuite([unittest.defaultTestLoader.loadTestsFromTestCase(CreditTests),unittest.defaultTestLoader.loadTestsFromTestCase(ContractTests)])
test_result=unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()
checks["synthetic_tests_passed"]=test_result.wasSuccessful()
checks["synthetic_test_count"]=test_result.testsRun
assert all(v for k,v in checks.items() if k!="synthetic_test_count"),checks
(OUT/"validation.json").write_text(json.dumps(checks,indent=2),encoding="utf-8")
environment={"python":platform.python_version(),"pandas":pd.__version__,"numpy":np.__version__,"matplotlib":matplotlib.__version__}
try:
    import sklearn
    environment["scikit-learn"]=sklearn.__version__
except ImportError: pass
(OUT/"environment.json").write_text(json.dumps(environment,indent=2),encoding="utf-8")
display(pd.DataFrame([environment]))
(OUT/"README.md").write_text("Root2Raj AI-assisted public-data experiment. Source: Chen, D. (2015), UCI Online Retail, DOI 10.24432/C5BW33, CC BY 4.0. Source workbook SHA-256: "+source_hash+"\nRead the notebook for assumptions, limitations and interpretation. Derived evidence is CC BY 4.0; code is Apache 2.0.\n",encoding="utf-8")
(ROOT/"requirements.txt").write_text("pandas>=2.2,<4\nnumpy>=1.26,<3\nopenpyxl>=3.1,<4\nmatplotlib>=3.8,<4\nscikit-learn>=1.4,<2\nthreadpoolctl>=3.1,<4\n",encoding="utf-8")
package=ROOT/"root2raj-credit-linkage-uncertainty-evidence.zip"
with zipfile.ZipFile(package,"w",zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):
        if path.is_file(): z.write(path,"evidence/"+path.name)
    for name in ['credit_linkage.py','test_credit_linkage.py',"requirements.txt","evidence_contracts.py","test_evidence_contracts.py"]:
        z.write(ROOT/name,"source/"+name)
display(pd.DataFrame([checks]))
display(FileLink(str(package)))
print("Evidence includes aggregate tables, results, validation, environment, charts, source and tests.")